In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

url_menor_evasao = "https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Sketch/Filtrados/Tabelas/Tabela_Censo_Escolar_Menor_Evasao_2024.csv"
url_maior_evasao = "https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Sketch/Filtrados/Tabelas/Tabela_Censo_Escolar_Maior_Evasao_2024.csv"

df_menor_evasao = pd.read_csv(url_menor_evasao, sep=';')
df_maior_evasao = pd.read_csv(url_maior_evasao, sep=';')

colunas_infra = [
    'IN_AGUA_POTAVEL', 'IN_ENERGIA_REDE_PUBLICA', 'IN_ESGOTO_REDE_PUBLICA',
    'IN_BANHEIRO', 'IN_COZINHA', 'IN_REFEITORIO', 'IN_QUADRA_ESPORTES',
    'IN_LABORATORIO_CIENCIAS', 'IN_LABORATORIO_INFORMATICA', 'IN_SALA_DIRETORIA',
    'IN_SECRETARIA', 'IN_SALA_MULTIUSO', 'IN_SALA_LEITURA', 'IN_ALIMENTACAO',
    'IN_INTERNET', 'IN_INTERNET_ALUNOS'
]

df_menor_evasao['MEDIA_INFRAESTRUTURA'] = df_menor_evasao[colunas_infra].mean(axis=1) * 100
df_maior_evasao['MEDIA_INFRAESTRUTURA'] = df_maior_evasao[colunas_infra].mean(axis=1) * 100

print("Bases de dados carregadas e médias de infraestrutura calculadas.")

labels_maior = ['Q1 (Crítico - Piores 25%)', 'Q2 (Básico - 25% a 50%)',
                'Q3 (Intermediário - 50% a 75%)', 'Q4 (Melhores 25%)']
df_maior_evasao['QUARTIL_INFRA'] = pd.qcut(df_maior_evasao['MEDIA_INFRAESTRUTURA'], q=4, labels=labels_maior, duplicates='drop')
resumo_maior = df_maior_evasao.groupby('QUARTIL_INFRA', observed=False)['MEDIA_INFRAESTRUTURA'].mean().round(1).reset_index()

labels_menor = ['Q1 (Básico - Piores 25%)', 'Q2 (Intermediário - 25% a 50%)',
                'Q3 (Avançado - 50% a 75%)', 'Q4 (Excelência - Melhores 25%)']
df_menor_evasao['QUARTIL_INFRA'] = pd.qcut(df_menor_evasao['MEDIA_INFRAESTRUTURA'], q=4, labels=labels_menor, duplicates='drop')
resumo_menor = df_menor_evasao.groupby('QUARTIL_INFRA', observed=False)['MEDIA_INFRAESTRUTURA'].mean().round(1).reset_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('<b>MAIOR Evasão Escolar</b>', '<b>MENOR Evasão Escolar</b>'),
    horizontal_spacing=0.15
)

fig.add_trace(go.Bar(
    y=resumo_maior['QUARTIL_INFRA'],
    x=resumo_maior['MEDIA_INFRAESTRUTURA'],
    orientation='h',
    marker=dict(color=['#fcae91', '#fb6a4a', '#de2d26', '#a50f15']),
    text=[f"<b>{val}%</b>" for val in resumo_maior['MEDIA_INFRAESTRUTURA']],
    textposition='auto',
    hoverinfo='x+y',
    name='Maior Evasão'
), row=1, col=1)

fig.add_trace(go.Bar(
    y=resumo_menor['QUARTIL_INFRA'],
    x=resumo_menor['MEDIA_INFRAESTRUTURA'],
    orientation='h',
    marker=dict(color=['#bdd7e7', '#6baed6', '#3182bd', '#08519c']),
    text=[f"<b>{val}%</b>" for val in resumo_menor['MEDIA_INFRAESTRUTURA']],
    textposition='auto',
    hoverinfo='x+y',
    name='Menor Evasão'
), row=1, col=2)

cor_fundo = '#F8F9FA' # Off-White padrão do relatório

fig.update_layout(
    title=dict(text='<b>Impacto da Infraestrutura: Comparativo de Quartis</b>', x=0.5, font=dict(size=20, color='#111111')),
    showlegend=False,
    height=500,

    plot_bgcolor=cor_fundo,
    paper_bgcolor=cor_fundo,

    margin=dict(l=10, r=10, t=80, b=50),
    font=dict(color='#333333')
)

fig.update_xaxes(title_text='<b>Média de Infraestrutura (%)</b>', range=[0, 100], showgrid=True, gridcolor='#EBEBEB', row=1, col=1)
fig.update_xaxes(title_text='<b>Média de Infraestrutura (%)</b>', range=[0, 100], showgrid=True, gridcolor='#EBEBEB', row=1, col=2)

fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=1, col=2)

fig.show()